In [1]:
import zipfile
# Dùng trực tiếp đường dẫn và quy tắc tìm dữ liệu của notebook profiling revised.
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import uuid

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    PROJECT_PATH = Path('/content/drive/MyDrive/Data_Lakehouse_Project')
else:
    IN_COLAB = False
    current = Path.cwd()
    candidates = [current, *(parent for parent in current.parents)]
    PROJECT_PATH = next((p.resolve() for p in candidates if (p / 'data').is_dir()), current.resolve())

DATA_DIRS = [PROJECT_PATH / 'data' / 'raw', PROJECT_PATH / 'data' / 'bronze', PROJECT_PATH / 'data']
EXPECTED_FILES = [
    'application_train.csv', 'application_test.csv', 'bureau.csv', 'bureau_balance.csv',
    'previous_application.csv', 'POS_CASH_balance.csv', 'credit_card_balance.csv',
    'installments_payments.csv'
]

def find_file(filename):
    for directory in DATA_DIRS:
        candidate = directory / filename
        if candidate.is_file():
            return candidate
    data_root = PROJECT_PATH / 'data'
    matches = list(data_root.rglob(filename)) if data_root.is_dir() else []
    if matches:
        return matches[0]
    return None

SOURCE_FILES = {filename: find_file(filename) for filename in EXPECTED_FILES}
missing = [filename for filename, path in SOURCE_FILES.items() if path is None]
if missing:
    checked = ', '.join(str(path) for path in DATA_DIRS)
    raise FileNotFoundError(f'Không tìm thấy các CSV: {missing}. Đã kiểm tra các vị trí: {checked}')

BRONZE_DIR = PROJECT_PATH / 'data' / 'bronze'
MANIFEST_DIR = PROJECT_PATH / 'data' / 'manifests'
LOG_DIR = PROJECT_PATH / 'logs'
PREFERRED_DB = PROJECT_PATH / 'data' / 'lakehouse_db' / 'credit_lakehouse.duckdb'
LEGACY_DB = PROJECT_PATH / 'database' / 'credit_lakehouse.duckdb'
DB_PATH = PREFERRED_DB if PREFERRED_DB.exists() or not LEGACY_DB.exists() else LEGACY_DB
TEMP_DIR = PROJECT_PATH / 'tmp' / 'duckdb'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
DBT_PROFILES_DIR = PROJECT_PATH / 'tmp' / f'dbt_profiles_{RUN_ID}'
DBT_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
profile_text = (
    'credit_lakehouse:\n  target: local\n  outputs:\n    local:\n'
    '      type: duckdb\n      path: ' + json.dumps(DB_PATH.as_posix()) + '\n      threads: 4\n'
)
(DBT_PROFILES_DIR / 'profiles.yml').write_text(profile_text, encoding='utf-8')
DBT_TARGET_DIR = (Path('/content') / f'dbt_target_{RUN_ID}') if IN_COLAB else (PROJECT_PATH / 'tmp' / f'dbt_target_{RUN_ID}')
DBT_TARGET_DIR.mkdir(parents=True, exist_ok=True)
DBT_DIR = PROJECT_PATH / 'dbt'
REQUIRED_DBT_FILES = [
    'dbt_project.yml', 'models/sources/sources.yml', 'models/staging/staging.yml',
    'models/staging/stg_application_train.sql', 'models/staging/stg_application_test.sql',
    'models/staging/stg_bureau.sql', 'models/staging/stg_bureau_balance.sql',
    'models/staging/stg_previous_application.sql', 'models/staging/stg_pos_cash_balance.sql',
    'models/staging/stg_credit_card_balance.sql', 'models/staging/stg_installments_payments.sql',
    'models/intermediate/int_application_train_prep.sql',
    'models/marts/core/fct_application_train.sql',
    'models/marts/features/feature_mart_application_train.sql',
    'macros/generate_schema_name.sql'
]
missing_dbt = [name for name in REQUIRED_DBT_FILES if not (DBT_DIR / name).is_file()]
DBT_BUNDLE = PROJECT_PATH / 'dbt_models_bundle.zip'
if missing_dbt and DBT_BUNDLE.is_file():
    with zipfile.ZipFile(DBT_BUNDLE) as archive:
        archive.extractall(PROJECT_PATH)
    missing_dbt = [name for name in REQUIRED_DBT_FILES if not (DBT_DIR / name).is_file()]
if missing_dbt:
    raise FileNotFoundError(
        f'Thiếu {len(missing_dbt)} file model dbt: {missing_dbt}. '
        f'Hãy đặt dbt_models_bundle.zip tại {PROJECT_PATH} rồi chạy lại cell này.'
    )
os.environ.setdefault('DUCKDB_TEMP_DIRECTORY', str(TEMP_DIR))
os.environ.setdefault('DUCKDB_MEMORY_LIMIT', '4GB')
os.environ.setdefault('DUCKDB_THREADS', '4')
print('Thư mục dự án:', PROJECT_PATH)
print('Nguồn CSV:', {name: str(path) for name, path in SOURCE_FILES.items()})
print('Database:', DB_PATH)
print('run_id:', RUN_ID)

Mounted at /content/drive
Thư mục dự án: /content/drive/MyDrive/Data_Lakehouse_Project
Nguồn CSV: {'application_train.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/application_train.csv', 'application_test.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/application_test.csv', 'bureau.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/bureau.csv', 'bureau_balance.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/bureau_balance.csv', 'previous_application.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/previous_application.csv', 'POS_CASH_balance.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/POS_CASH_balance.csv', 'credit_card_balance.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/credit_card_balance.csv', 'installments_payments.csv': '/content/drive/MyDrive/Data_Lakehouse_Project/data/raw/installments_payments.csv'}
Database: /content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lak

In [2]:
import shutil
import site
import sysconfig
# Cài thư viện rồi thêm thư mục Scripts/bin của Python vào PATH để tìm được lệnh dbt.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'duckdb==1.5.6', 'dbt-core==1.9.4', 'dbt-duckdb==1.9.4'], check=True)
script_dirs = [
    Path(sysconfig.get_path('scripts')),
    Path(site.getuserbase()) / 'bin',
    Path(sys.executable).parent / 'Scripts',
    Path(sys.executable).parent / 'bin',
]
for script_dir in script_dirs:
    if script_dir.is_dir():
        os.environ['PATH'] = str(script_dir) + os.pathsep + os.environ.get('PATH', '')
DBT_EXECUTABLE = shutil.which('dbt')
if DBT_EXECUTABLE is None:
    for script_dir in script_dirs:
        for candidate in (script_dir / 'dbt', script_dir / 'dbt.exe'):
            if candidate.is_file():
                DBT_EXECUTABLE = str(candidate)
                break
        if DBT_EXECUTABLE:
            break
if DBT_EXECUTABLE is None:
    raise FileNotFoundError('Đã cài dbt-core nhưng không tìm thấy chương trình dbt trong Scripts/bin của Python. Hãy chạy lại cell cài thư viện hoặc khởi động lại runtime rồi chạy từ đầu.')
print('dbt command:', DBT_EXECUTABLE)

dbt command: /usr/local/bin/dbt


In [3]:
# Nạp Bronze trực tiếp trong notebook — không phụ thuộc file .py bên ngoài.
import hashlib
import logging
import os
import duckdb


for folder in (BRONZE_DIR, MANIFEST_DIR, LOG_DIR, DB_PATH.parent, TEMP_DIR):
    folder.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.FileHandler(LOG_DIR / f'ingest_{RUN_ID}.log', encoding='utf-8'), logging.StreamHandler()], force=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def sql_path(path):
    return str(path.resolve()).replace("'", "''")

manifest = {'run_id': RUN_ID, 'loaded_at_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_PATH), 'execution_environment': 'Google Colab' if IN_COLAB else 'Local Jupyter', 'sources': []}
con = duckdb.connect(str(DB_PATH))
try:
    con.execute(f"SET memory_limit='{os.environ.get('DUCKDB_MEMORY_LIMIT', '4GB')}'")
    con.execute(f"SET threads={int(os.environ.get('DUCKDB_THREADS', '4'))}")
    con.execute(f"SET temp_directory='{sql_path(TEMP_DIR)}'")
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
    for filename in sorted(EXPECTED_FILES):
        source = SOURCE_FILES[filename]
        source_hash = sha256_file(source)
        table = source.stem.lower()
        batch_dir = BRONZE_DIR / table / f'source_sha256={source_hash}'
        target = batch_dir / 'data.parquet'
        batch_dir.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            temp_file = batch_dir / f'.data.{RUN_ID}.tmp.parquet'
            try:
                con.execute(f"COPY (SELECT * FROM read_csv('{sql_path(source)}', header=true, all_varchar=true)) TO '{sql_path(temp_file)}' (FORMAT PARQUET)")
                os.replace(temp_file, target)
            finally:
                if temp_file.exists():
                    temp_file.unlink()
        parquet_path = sql_path(target)
        rows = con.execute(f"SELECT count(*) FROM read_parquet('{parquet_path}')").fetchone()[0]
        con.execute(f"CREATE OR REPLACE VIEW bronze.{table} AS SELECT * FROM read_parquet('{parquet_path}')")
        manifest['sources'].append({'file': filename, 'source_sha256': source_hash,
            'parquet': target.relative_to(PROJECT_PATH).as_posix(), 'duckdb_relation': f'bronze.{table}',
            'row_count': rows, 'source_bytes': source.stat().st_size, 'parquet_bytes': target.stat().st_size})
        logging.info('%s: %s rows; Parquet %.1f MiB', filename, f'{rows:,}', target.stat().st_size / 1024**2)
except Exception:
    logging.exception('Bronze ingestion failed for run_id=%s', RUN_ID)
    raise
finally:
    con.close()

manifest_path = MANIFEST_DIR / f'{RUN_ID}.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
latest_temp = MANIFEST_DIR / f'.latest.{RUN_ID}.tmp'
latest_temp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
os.replace(latest_temp, MANIFEST_DIR / 'latest.json')
print('Bronze complete:', len(manifest['sources']), 'sources')
print('Database:', DB_PATH)
print('Manifest:', manifest_path)

2026-10-27 07:10:08,844 INFO POS_CASH_balance.csv: 10,001,358 rows; Parquet 142.3 MiB
2026-10-27 07:10:13,107 INFO application_test.csv: 48,744 rows; Parquet 4.9 MiB
2026-10-27 07:10:19,289 INFO application_train.csv: 307,511 rows; Parquet 29.0 MiB
2026-10-27 07:10:26,001 INFO bureau.csv: 1,716,428 rows; Parquet 45.2 MiB
2026-10-27 07:10:39,294 INFO bureau_balance.csv: 27,299,925 rows; Parquet 17.1 MiB
2026-10-27 07:10:52,685 INFO credit_card_balance.csv: 3,840,312 rows; Parquet 165.9 MiB
2026-10-27 07:11:13,088 INFO installments_payments.csv: 13,605,401 rows; Parquet 404.2 MiB
2026-10-27 07:11:25,057 INFO previous_application.csv: 1,670,214 rows; Parquet 84.6 MiB


Bronze complete: 8 sources
Database: /content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb
Manifest: /content/drive/MyDrive/Data_Lakehouse_Project/data/manifests/20261027T070937Z_78cf1c6d.json


In [7]:
# Xác nhận dbt nhận diện đủ model rồi chạy vào target mới của lần này.
EXPECTED_DBT_MODELS = {
    'stg_application_train', 'stg_application_test', 'stg_bureau', 'stg_bureau_balance',
    'stg_previous_application', 'stg_pos_cash_balance', 'stg_credit_card_balance',
    'stg_installments_payments', 'int_application_train_prep',
    'fct_application_train', 'feature_mart_application_train'
}
listing = subprocess.run(
    [DBT_EXECUTABLE, 'ls', '--resource-type', 'model', '--output', 'name', '--quiet',
     '--profiles-dir', str(DBT_PROFILES_DIR), '--target-path', str(DBT_TARGET_DIR)],
    cwd=DBT_DIR, check=True, capture_output=True, text=True
)
discovered_models = {line.strip() for line in listing.stdout.splitlines() if line.strip()}
missing_models = sorted(EXPECTED_DBT_MODELS - discovered_models)
print('dbt nhận diện', len(discovered_models), 'model:', sorted(discovered_models))
if missing_models:
    raise RuntimeError(f'dbt chưa nhận diện đủ model; còn thiếu: {missing_models}. Kiểm tra thư mục dbt/models trên Drive.')

subprocess.run(
    [DBT_EXECUTABLE, 'build', '--profiles-dir', str(DBT_PROFILES_DIR), '--target-path', str(DBT_TARGET_DIR)],
    cwd=DBT_DIR, check=True
)

dbt nhận diện 11 model: ['fct_application_train', 'feature_mart_application_train', 'int_application_train_prep', 'stg_application_test', 'stg_application_train', 'stg_bureau', 'stg_bureau_balance', 'stg_credit_card_balance', 'stg_installments_payments', 'stg_pos_cash_balance', 'stg_previous_application']


CompletedProcess(args=['/usr/local/bin/dbt', 'build', '--profiles-dir', '/content/drive/MyDrive/Data_Lakehouse_Project/tmp/dbt_profiles_20261027T070937Z_78cf1c6d', '--target-path', '/content/dbt_target_20261027T070937Z_78cf1c6d'], returncode=0)

In [8]:
# Tổng hợp kết quả và dò schema thực tế do dbt tạo.
import shutil

con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    relations = con.execute("""
        select table_schema, table_name
        from information_schema.tables
        where table_schema not in ('information_schema', 'pg_catalog')
    """).fetchall()
    def count_model(table_name, preferred_schema):
        matches = [(schema, name) for schema, name in relations if name == table_name]
        if not matches:
            return None, None
        matches.sort(key=lambda item: (item[0] != preferred_schema, not item[0].endswith('_' + preferred_schema), item[0]))
        schema, name = matches[0]
        quoted_schema = schema.replace('"', '""')
        quoted_name = name.replace('"', '""')
        count = con.execute(f'SELECT COUNT(*) FROM "{quoted_schema}"."{quoted_name}"').fetchone()[0]
        return count, f'{schema}.{name}'

    train_rows, train_relation = count_model('stg_application_train', 'silver')
    test_rows, test_relation = count_model('stg_application_test', 'silver')
    fact_rows, fact_relation = count_model('fct_application_train', 'gold')
    feature_rows, feature_relation = count_model('feature_mart_application_train', 'gold')
    results = {
        'run_id': RUN_ID,
        'database': str(DB_PATH),
        'execution_environment': 'Google Colab' if IN_COLAB else 'Local Jupyter',
        'bronze_sources': len(manifest['sources']),
        'bronze_rows': {item['file']: item['row_count'] for item in manifest['sources']},
        'silver_application_train_rows': train_rows,
        'silver_application_train_relation': train_relation,
        'silver_application_test_rows': test_rows,
        'silver_application_test_relation': test_relation,
        'gold_application_fact_rows': fact_rows,
        'gold_application_fact_relation': fact_relation,
        'gold_feature_mart_rows': feature_rows,
        'gold_feature_mart_relation': feature_relation,
        'dbt_models_complete': all(value is not None for value in [train_rows, test_rows, fact_rows, feature_rows]),
        'available_relations': [f'{schema}.{name}' for schema, name in sorted(relations)],
    }
finally:
    con.close()

EVIDENCE_DIR = PROJECT_PATH / 'evidence' / RUN_ID
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
(EVIDENCE_DIR / 'pipeline_summary.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
run_results = DBT_TARGET_DIR / 'run_results.json'
if run_results.is_file():
    dbt_result_doc = json.loads(run_results.read_text(encoding='utf-8'))
    statuses = {}
    for item in dbt_result_doc.get('results', []):
        status = item.get('status', 'unknown')
        statuses[status] = statuses.get(status, 0) + 1
    results['dbt_status_counts'] = statuses
    results['dbt_invocation_id'] = dbt_result_doc.get('metadata', {}).get('invocation_id')
    shutil.copy2(run_results, EVIDENCE_DIR / 'dbt_run_results.json')
print(json.dumps(results, indent=2))
if not results['dbt_models_complete']:
    print('Cảnh báo: chưa tìm thấy đủ các model Silver/Gold. Xem available_relations và dbt_run_results.json để biết dbt đã tạo bảng nào.')
print('Evidence:', EVIDENCE_DIR)

{
  "run_id": "20261027T070937Z_78cf1c6d",
  "database": "/content/drive/MyDrive/Data_Lakehouse_Project/database/credit_lakehouse.duckdb",
  "execution_environment": "Google Colab",
  "bronze_sources": 8,
  "bronze_rows": {
    "POS_CASH_balance.csv": 10001358,
    "application_test.csv": 48744,
    "application_train.csv": 307511,
    "bureau.csv": 1716428,
    "bureau_balance.csv": 27299925,
    "credit_card_balance.csv": 3840312,
    "installments_payments.csv": 13605401,
    "previous_application.csv": 1670214
  },
  "silver_application_train_rows": 307511,
  "silver_application_train_relation": "silver.stg_application_train",
  "silver_application_test_rows": 48744,
  "silver_application_test_relation": "silver.stg_application_test",
  "gold_application_fact_rows": 307511,
  "gold_application_fact_relation": "gold.fct_application_train",
  "gold_feature_mart_rows": 307511,
  "gold_feature_mart_relation": "gold.feature_mart_application_train",
  "dbt_models_complete": true,
  "avai